### Supplement: OpenAI API Example & Persona Deep Dive

This supplement notebook breaks down `openai_api_example.ipynb` cell by cell. It demystifies:

1. **The Texan Persona & Chat Roles**: How `system` messages configure dialect and personality, and how `user` messages trigger responses.
2. **Class vs. Object in the SDK**:
   - `OpenAI` is the Client Class; `client` is the configured Object.
   - `ChatCompletion` is the Response Schema Class; `completion` is the deserialized Object.
3. **Deconstructing the Response Chain**: Detailed traversal of `response = completion.choices[0].message.content`.
4. **Wire Format vs. Python Object**: How raw HTTP JSON becomes a `ChatCompletion` Pydantic model and how to view it as a dictionary (`.model_dump()`).
5. **Multi-turn Memory with the Texan Persona**: Continuing the conversation while preserving the persona by appending `role: 'assistant'`.

---

##### Architectural Diagram:
```
client = OpenAI(...)  <-- [Object of OpenAI class]
   │
   ├── messages = [
   │     {'role': 'system', 'content': 'Imagine you are a Texan'},   <-- Persona/Rules
   │     {'role': 'user', 'content': 'Give me information about SpaceX'}  <-- User Query
   │   ]
   │
   └── completion = client.chat.completions.create(...)  <-- HTTP POST to OpenAI REST API
         │
         ▼
       ChatCompletion (Pydantic Object)
         ├── id: 'chatcmpl-...'
         ├── usage: { prompt_tokens, completion_tokens, total_tokens }
         └── choices: [
               Choice(
                 index=0,
                 finish_reason='stop',
                 message=ChatCompletionMessage(
                   role='assistant',   <-- Model's spoken turn
                   content='Howdy! ...'   <-- The generated reply
                 )
               )
             ]
```


#### 1. Imports and Setup

We load environment variables and import the `OpenAI` client class.

In [1]:
import os
import json
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# Automatically locate and load .env
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(r"C:\Users\ashut\ML_Practice\LLM_Learning_Sandbox\.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Client instance created successfully.")
print("Client class:", type(client))
print("Base API URL:", client.base_url)


Client instance created successfully.
Client class: <class 'openai.OpenAI'>
Base API URL: https://api.openai.com/v1/


#### 2. Defining the Messages: Persona vs. Query

##### Why `role: 'system'`?
The `system` message acts as invisible scaffolding for the conversation. It tells the model *how* to speak, *what* tone to use, and *what* boundaries to respect. Here, it instructs the model: `"Imagine you are a Texan"`.

The `user` message is the actual task: `"Give me information about SpaceX"`.

In [2]:
messages = [
    {'role': 'system', 'content': 'Imagine you are a Texan'}, 
    {'role': 'user', 'content': 'Give me information about SpaceX'}
]

print("Messages list:")
print(json.dumps(messages, indent=2))


Messages list:
[
  {
    "role": "system",
    "content": "Imagine you are a Texan"
  },
  {
    "role": "user",
    "content": "Give me information about SpaceX"
  }
]


#### 3. Executing the API Call

We call `client.chat.completions.create(...)`. Under the hood, this sends an HTTP `POST` request to `https://api.openai.com/v1/chat/completions` with our messages payload.

The response from the server is deserialized into a `ChatCompletion` object.

In [3]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages
)

print("Call completed!")
print("Type of completion:", type(completion))
print("Completion ID:", completion.id)
print("Model used:", completion.model)
print("Token usage summary:", completion.usage)


Call completed!
Type of completion: <class 'openai.types.chat.chat_completion.ChatCompletion'>
Completion ID: chatcmpl-EOf1Bpwd6SpWgW5OIFhqiLOAFoUTR
Model used: gpt-5-nano-2025-08-07
Token usage summary: CompletionUsage(completion_tokens=3568, prompt_tokens=22, total_tokens=3590, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=3008, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


#### 4. Deconstructing `response = completion.choices[0].message.content`

Let's inspect every attribute along the chain to understand what data type and information each part carries:

In [4]:
print("=== Level 1: completion.choices ===")
print("Type:", type(completion.choices))
print("Length:", len(completion.choices))

print("\n=== Level 2: completion.choices[0] (Choice object) ===")
first_choice = completion.choices[0]
print("Type:", type(first_choice))
print("Finish reason:", first_choice.finish_reason)
print("Index:", first_choice.index)

print("\n=== Level 3: completion.choices[0].message (ChatCompletionMessage object) ===")
message = first_choice.message
print("Type:", type(message))
print("Role:", repr(message.role), "| Type:", type(message.role))

print("\n=== Level 4: completion.choices[0].message.content (Generated string) ===")
response = message.content
print("Type of response:", type(response))
print("-" * 50)
print("Preview of Texan Response (first 400 chars):")
print(response[:400] + "...\n[truncated for preview]")
print("-" * 50)


=== Level 1: completion.choices ===
Type: <class 'list'>
Length: 1

=== Level 2: completion.choices[0] (Choice object) ===
Type: <class 'openai.types.chat.chat_completion.Choice'>
Finish reason: stop
Index: 0

=== Level 3: completion.choices[0].message (ChatCompletionMessage object) ===
Type: <class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
Role: 'assistant' | Type: <class 'str'>

=== Level 4: completion.choices[0].message.content (Generated string) ===
Type of response: <class 'str'>
--------------------------------------------------
Preview of Texan Response (first 400 chars):
Howdy, friend. Here’s the quick lay of SpaceX and what they’re all about.

- What SpaceX is
  - An American aerospace company founded in 2002 by Elon Musk. Their big goal is to cut the cost of spaceflight and, one day, help humans live on other worlds (think Mars).

- Key rockets and capsules
  - Falcon 9: Reusable first stage that can land back on a ship or pad. Used for hauling satell

#### 5. Visualizing the Full Response as a Python Dictionary

By calling `.model_dump()`, we convert the Pydantic object into a standard nested Python dictionary. This makes it easy to visualize the complete JSON response returned by the API:

In [5]:
response_dict = completion.model_dump()

print("Type of completion.model_dump():", type(response_dict))
print("Top-level keys:", list(response_dict.keys()))

print("\n--- Full Nested Response (Formatted JSON) ---")
print(json.dumps(response_dict, indent=2))


Type of completion.model_dump(): <class 'dict'>
Top-level keys: ['id', 'choices', 'created', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage']

--- Full Nested Response (Formatted JSON) ---
{
  "id": "chatcmpl-EOf1Bpwd6SpWgW5OIFhqiLOAFoUTR",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Howdy, friend. Here\u2019s the quick lay of SpaceX and what they\u2019re all about.\n\n- What SpaceX is\n  - An American aerospace company founded in 2002 by Elon Musk. Their big goal is to cut the cost of spaceflight and, one day, help humans live on other worlds (think Mars).\n\n- Key rockets and capsules\n  - Falcon 9: Reusable first stage that can land back on a ship or pad. Used for hauling satellites, cargo to the ISS, and crew launches.\n  - Falcon Heavy: Heavy-lift variant of Falcon 9 with more payload capacity.\n  - Dragon: Spacecraft that can carry cargo (Dragon 1/2 for cargo) and crew (Crew 

#### 6. Multi-Turn Conversation: Why `role: 'assistant'` Matters

##### Maintaining the Texan Persona Across Turns
When you have a conversation with ChatGPT, how does it remember what was said before?
**The OpenAI API is stateless!** It remembers nothing between API calls unless *you* pass the conversation history.

To continue the conversation:
1. Append the model's previous reply to `messages` as `{"role": "assistant", "content": response}`.
2. Append your new question as `{"role": "user", "content": "Now tell me about Blue Origin too"}`.
3. Send the updated `messages` list to the API.

Because the reply is tagged as `assistant` and the original system prompt is still present, the model knows:
- "I previously told the user about SpaceX."
- "I am still a Texan."
- "Now the user wants to know about Blue Origin."

Let's test this in code:

In [6]:
# Reset to base 2 turns so rerunning this cell remains idempotent
messages = messages[:2]

# 1. Append assistant response to history
messages.append({'role': 'assistant', 'content': response})

# 2. Append new user query
messages.append({'role': 'user', 'content': 'Now tell me about Blue Origin too, and keep your Texan style!'})

print(f"Conversation history now has {len(messages)} messages.")
print("Roles in sequence:", [m['role'] for m in messages])

# 3. Call API with full conversation context
followup_completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages
)

followup_response = followup_completion.choices[0].message.content
print("\n=== Texan Follow-up on Blue Origin ===")
print(followup_response)


Conversation history now has 4 messages.
Roles in sequence: ['system', 'user', 'assistant', 'user']

=== Texan Follow-up on Blue Origin ===
Howdy, partner. Here’s the lay of Blue Origin, Texan-style:

- What Blue Origin is
  - An American aerospace company started in 2000 by Jeff Bezos. They’re all about enabling private spaceflight, building reusable rockets, and laying the groundwork for a space-enabled economy—think suborbital joyrides, orbital heavy lifting, and future Moon business.

- Key rides and gear
  - New Shepard: Suborbital rocket and capsule system built for passenger flights to the edge of space and back. It’s designed to be reusable, with the booster landing upright and the capsule returning by parachute. Used for space tourism and microgravity research; the company has run numerous flights, including a crewed mission in 2021.
  - New Glenn: A big orbital-class rocket in development. It’s Blue Origin’s heavy-lift vehicle intended to carry large payloads to orbit and bey